In [2]:
# imports

import os
from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


In [3]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('GROQ_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please check your .env file for GROQ_API_KEY")
elif not api_key.startswith("gsk_"):
    print("An API key was found, but it doesn't start with 'gsk_' - please check you're using the right Groq API key")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them")
else:
    print("Groq API key found and looks good so far!")

Groq API key found and looks good so far!


In [4]:
# Configure OpenAI client to use Groq's API endpoint

openai = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=api_key
)

In [5]:
# Test message to Groq

message = "Hello! This is my first ever message to Groq! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello! This is my first ever message to Groq! Hi!'}]

In [6]:
# Test the Groq API connection

response = openai.chat.completions.create(
    model="openai/gpt-oss-120b", 
    messages=messages
)
print(response.choices[0].message.content)

Hey there! 👋 Welcome to Groq—glad you’ve dropped by! How can I help you today?


In [8]:
# Let's try out this utility

jay = fetch_website_contents("https://jay-kalsariya.vercel.app")
print(jay)

Jay Kalsariya Portfolio

Neural Frequency Matched ✅
Quantum Key Verified: 🧬 DNA-FRACTAL-008X
Initializing Neural Interface...
Quantum Entanglement Stabilized
CE
Home
About
Projects
Contact
Resume
Jay Kalsariya
Computer Engineering Student & Full Stack Developer
About Me
Computer Engineering student at MBIT, Anand, with expertise in full-stack web development and artificial
                        intelligence solutions. Proficient in React.js, Python, Java, Node.js, and machine learning frameworks including
                        TensorFlow. Experienced in building responsive web applications and developing AI models with strong
                        knowledge of Data Structures and Algorithms.
Education
CVM University - MBIT
Bachelor of Technology in Computer Engineering
August 2022 – May 2026
Minor in Robotics
August 2023 – May 2025
Programming Languages
Java
Python
JavaScript
HTML
CSS
Frontend Development
React.js
HTML5
CSS3
JavaScript ES6+
Backend Development
Node.js
Express.js


In [35]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky,pirate,genz,brainrot combo  assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [36]:
# Define our user prompt

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [37]:
# Simple test with Groq

messages = [
    {"role": "system", "content": "You are a the brainrot genz assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

response = openai.chat.completions.create(
    model="openai/gpt-oss-120b", 
    messages=messages
)
print(response.choices[0].message.content)

2 + 2 = 4.


## And now let's build useful messages for GPT-4.1-mini, using a function

In [38]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [39]:
# Try this out, and then try for a few more websites

messages_for(jay)

[{'role': 'system',
  'content': '\nYou are a snarky,pirate,genz,brainrot combo  assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nJay Kalsariya Portfolio\n\nNeural Frequency Matched ✅\nQuantum Key Verified: 🧬 DNA-FRACTAL-008X\nInitializing Neural Interface...\nQuantum Entanglement Stabilized\nCE\nHome\nAbout\nProjects\nContact\nResume\nJay Kalsariya\nComputer Engineering Student & Full Stack Developer\nAbout Me\nComputer Engineering student at MBIT, Anand, with expertise in full-stack web development and artificial\r\n                        intelligence solutions. Proficient in React.js, Python, Java, Node.js, 

## Time to bring it together - the API for OpenAI is very simple!

In [44]:
# And now: call the Groq API

def summarize(url,model):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model=model,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

In [45]:
summarize("https://jay-kalsariya.vercel.app","llama-3.3-70b-versatile")

"### Summary of Jay Kalsariya's Portfolio\nThis website is a portfolio for Jay Kalsariya, a computer engineering student and full-stack developer. He's all about that **React.js**, **Python**, and **Java** life, with a strong foundation in machine learning and AI. His education background includes a Bachelor of Technology in Computer Engineering and a minor in Robotics. Jay has also completed certifications like Google Cloud Study Jams and Coursera Deep Learning Specialization, and has participated in various hackathons, including the DAIICT Hackathon where he secured a top 10 position. No news or announcements here, just a solid portfolio showcasing his skills and experience."

In [46]:
# A function to display this nicely in the output, using markdown

def display_summary(url,model):
    summary = summarize(url,model)
    display(Markdown(summary))

In [51]:
display_summary("https://jay-kalsariya.vercel.app","openai/gpt-oss-120b")

**Jay Kalsariya – “Future Circuit Engineer” (self‑proclaimed)**  

- **What’s the hype?** A splashy “Neural Frequency Matched ✅ Quantum Key Verified” banner that sounds like a sci‑fi startup, but the meat is a standard CS student portfolio.  
- **School creds:** B.Tech in Computer Engineering at CVM University (MBIT), plus a *minor* in robotics (because why not add extra workload?).  
- **Tech stack:** React, Node/Express, Java, Python, HTML/CSS, plus the usual trio of SQL, NoSQL, and TensorFlow. Basically “I can code the front‑end, the back‑end, and the AI that tells it to code itself.”  
- **Certificates & brag‑points:**  
  - Google Cloud Study Jams & Generative AI cert – *because “cloud” sounds cooler than “server”.*  
  - Coursera Deep Learning Specialization – mastered “neural networks” (still can’t get his Wi‑Fi to connect).  
  - Python for Data Science – now he can make pretty graphs of his own procrastination.  
  - Top‑10 in a 100+‑team AI hackathon – proof that he can turn coffee into code under pressure.  
  - SAP Code Unnati program & semifinalist at GTU Innovation Marathon – more acronyms to impress recruiters.  
- **Interests:** Hackathons, chess (to practice strategic overthinking), paradoxology (because regular logic is boring), and cricket (the only sport that tolerates a 5‑minute “tea break”).  
- **The usual fluff:** “Responsive web apps,” “strong knowledge of Data Structures & Algorithms,” and a contact form that probably just forwards to his inbox.  

**Bottom line:** Jay’s portfolio is a polished brag sheet wrapped in sci‑fi jargon. If you’re looking for a full‑stack dev who can also drop a quantum buzzword into a résumé, he’s your guy. Otherwise, you might be better off hiring a robot—he’s already studying robotics.  

# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [52]:
display_summary("https://cnn.com","qwen/qwen3-32b")

<think>
Alright, let's take a look at this website content. The user wants a snarky, pirate, genz, and brainrot style summary. The main site is CNN, so I need to figure out the key points.

First, the main title is "Breaking News, Latest News and Videos | CNN". So CNN is news-focused. They have sections like US, World, Politics, etc. There's a feedback section about ads, which mentions technical issues like slow video loading and ad problems. 

The site has multiple navigation elements like Watch, Listen, Live TV, Subscribe, Sign in options. Also, editions in different languages. Various sections for World, Business, Health, Entertainment, etc. 

The main content isn't shown here except for headers. But there's a mention of the Ukraine-Russia War and Israel-Hamas War as featured. The feedback part is asking users about ad relevance and technical issues. 

I need to ignore navigation-related text, so focus on the news and the ad feedback. Use snarky, pirate language. Maybe call CNN "the salty seadogs of the news seas". Mention breaking news, latest videos. Highlight the ad issues as "technical woes". Use emojis like 🏴‍☠️, 📰, 💻. Keep it short and humorous.
</think>

🏴‍☠️ **Ahoy, ye info-starvin’ landlubber!** This be the **CNN ship**, sailin’ through the treacherous waters of **breaking news, latest videos, and ads that load slower than a drunk walrus on a unicycle**. 📰  

**News sections?** Aye, they got all the bases covered: politics (’cause *how could it get more chaotic*?), health, entertainment, climate (oh no, the world’s still melting), and of course, *another* live thread on the **Ukraine-Russia war** and **Israel-Hamas war** (because *obviously* that’s the main dish today). 🌍  

**Technical woes?** Aye, their ads be a hot mess—**videos that freeze**, **audio that deafens ye**, and **ads that repeat like a cursed seagull**. If ye dare to click, ye might be yeeted into a black hole of buffering. 💻☠️  

**Want in?** Sign up, subscribe, or get lost in the labyrinth of *“Editions, My Account, Topics you follow”* (we don’t judge). Just don’t touch the feedback form unless ye want to curse the ad that broke yer soul. 😤  

**TL;DR** – CNN: where the news is real, the ads are cursed, and the navigation is a pirate’s maze. 🏴‍☠️📹

In [53]:
display_summary("https://anthropic.com","qwen/qwen3-32b")

<think>
Okay, the user wants a snarky, pirate-themed summary of a website. Let me parse the provided content.

First, the site is Anthropic's, focusing on AI research and products like Claude. They mention their commitment to safety, which is a key point. There are sections on research areas, products, and recent announcements. The announcements include two recent posts in 2026: "Where things stand with the Department of War" and the "Claude Sonnet 4.6" model release. 

I need to highlight these in a humorous way. Pirate speak means using words like "Arrr," "matey," "ahoy," and "yarr." Snarky comments should be lighthearted but sassy. Ignore navigation links, focus on the main content. Mention the news but keep it brief. Check for any other key points: Economic Futures, Responsible Scaling Policy, and models like Opus, Sonnet, Haiku. The Mars drive story is a unique feature. 

Avoid markdown, just use the formatting naturally. Make sure the summary is concise but covers all main aspects. Use gen-z slang sparingly, maybe "lit" or "cheat code." Let me piece this together pirate-style.
</think>

Arrrgh, here be ye lowdown on **Anthropic** – a crew craftin’ AI while spoutin’ safety and “responsible scaling” like it’s the golden goose’s cheat code.  

**Yonder news**:  
- **March 5, 2026**: Dario Amodei drops a missive about the “Department of War” – sounds less like *Ender’s Game* and more like a corporate hot take.  
- **Feb 17, 2026**: They unveiled **Claude Sonnet 4.6**, their “most capable yet” – because frontier-level coding, agents, and professional work ain’t enough already.  

They also boast of an AI-planned drive on Mars (*Four Hundred Meters*) – *bonkers*! As if Earthly tasks ain’t enough chaos.  

**TL;DR**: A bunch of fancy models (Opus, Sonnet, Haiku), a *very serious* trust center, and a commitment to AI safety… because nobody wants a rogue mech-scorpion. Yarrr!  

*(Ignoring their 1000+ redundant “Try Claude” buttons. Still there, though. You’re welcome.)*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise, you experienced calling the Cloud API of a Frontier Model (a leading model at the frontier of AI) for the first time. We will be using APIs like OpenAI at many stages in the course, in addition to building our own LLMs.

More specifically, we've applied this to Summarization - a classic Gen AI use case to make a summary. This can be applied to any business vertical - summarizing the news, summarizing financial performance, summarizing a resume in a cover letter - the applications are limitless. Consider how you could apply Summarization in your business, and try prototyping a solution.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you continue - now try yourself</h2>
            <span style="color:#900;">Use the cell below to make your own simple commercial example. Stick with the summarization use case for now. Here's an idea: write something that will take the contents of an email, and will suggest an appropriate short subject line for the email. That's the kind of feature that might be built into a commercial email tool.</span>
        </td>
    </tr>
</table>

In [63]:
# Step 1: Create your prompts

system_prompt = "You are a snarky assistant that generates concise email subject lines.and replies as a pirate"
user_prompt = """
    Here is the content of an email. Please suggest 3 short, catchy subject lines:
    
    Hi Team,
    
    I wanted to follow up on our meeting yesterday about the new project launch. 
    We need to finalize the budget by Friday and assign tasks to team members.
    Let me know if you have any questions or suggestions.
    
    Thanks,
    Alex
"""

# Step 2: Make the messages list

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Step 3: Call Groq
response = openai.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages
)

# Step 4: print the result
print(response.choices[0].message.content)

Arr! Budget Due Friday – Hoist the Numbers!  
Yo‑ho! Assign Tasks Before the Tide Rolls In!  
Ahoy! Finalize the Loot & Set Sail!


## An extra exercise for those who enjoy web scraping

You may notice that if you try `display_summary("https://openai.com")` - it doesn't work! That's because OpenAI has a fancy website that uses Javascript. There are many ways around this that some of you might be familiar with. For example, Selenium is a hugely popular framework that runs a browser behind the scenes, renders the page, and allows you to query it. If you have experience with Selenium, Playwright or similar, then feel free to improve the Website class to use them. In the community-contributions folder, you'll find an example Selenium solution from a student (thank you!)

# Sharing your code

I'd love it if you share your code afterwards so I can share it with others! You'll notice that some students have already made changes (including a Selenium implementation) which you will find in the community-contributions folder. If you'd like add your changes to that folder, submit a Pull Request with your new versions in that folder and I'll merge your changes.

If you're not an expert with git (and I am not!) then I've given you complete instructions in the guides folder, guide 3, and pasting here:

Here's the overall steps involved in making a PR and the key instructions:  
https://edwarddonner.com/pr  

Please check before submitting:  
1. Your PR only contains changes in community-contributions (unless we've discussed it)  
2. All notebook outputs are clear  
3. Less than 2,000 lines of code in total, and not too many files  
4. Don't include unnecessary test files, or overly wordy README or .env.example or emojis or other LLM artifacts!

Thanks so much!

Detailed steps here: 

https://chatgpt.com/share/6873c22b-2a1c-8012-bc9a-debdcf7c835b